In [2]:
import os
import glob
import json
import random

random.seed(42)

# Point this to wherever you extracted the dataset
BASE_DIR = "/Users/farhanwajid/Web Development/SIH/v_2"
OUTPUT_FILE = "Outputs/crossmodal_fusion.jsonl"

CATEGORIES = ["agri", "barrenland", "grassland", "urban"]

# Simple instruction templates — randomized for variety
TEMPLATES = [
    "Use the optical and SAR images together to identify the land cover type.",
    "Fuse the optical and radar imagery to describe what is visible in this region.",
    "Analyze the paired optical and SAR images and describe the dominant terrain.",
    "What type of land cover does this optical-SAR image pair represent?",
]

RESPONSE_MAP = {
    "agri": "This region shows agricultural land, characterized by organized crop patterns visible in the optical image and moderate radar backscatter in the SAR image.",
    "barrenland": "This region shows barren land, with sparse vegetation visible in the optical image and low, uniform radar backscatter in the SAR image.",
    "grassland": "This region shows grassland, with continuous low vegetation cover visible in the optical image and consistent moderate radar backscatter in the SAR image.",
    "urban": "This region shows a built-up urban area, with dense structures visible in the optical image and high radar backscatter due to man-made surfaces in the SAR image.",
}

def build_dataset():
    records = []
    rec_id = 0

    for category in CATEGORIES:
        s1_folder = os.path.join(BASE_DIR, category, "s1")
        s2_folder = os.path.join(BASE_DIR, category, "s2")

        s1_files = sorted(glob.glob(os.path.join(s1_folder, "*.png")))
        s2_files = sorted(glob.glob(os.path.join(s2_folder, "*.png")))

        if len(s1_files) != len(s2_files):
            print(f"⚠️ Mismatch in {category}: {len(s1_files)} SAR vs {len(s2_files)} Optical files")

        # Pair them up (assumes matching sorted order/filenames)
        for s1_path, s2_path in zip(s1_files, s2_files):
            record = {
                "id": f"crossmodal_{rec_id:06d}",
                "task": "fusion",
                "image_1": s2_path,  # Optical as primary
                "image_2": s1_path,  # SAR as secondary
                "instruction": random.choice(TEMPLATES),
                "response": RESPONSE_MAP[category],
                "bbox": None,
                "source_dataset": "Kaggle-Sentinel12",
                "terrain_label": category  # keep this for stratified sampling later
            }
            records.append(record)
            rec_id += 1

    return records

if __name__ == "__main__":
    records = build_dataset()
    print(f"Total pairs converted: {len(records)}")

    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    with open(OUTPUT_FILE, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")

    print(f"✅ Saved to {OUTPUT_FILE}")

    # Quick sanity check: print one example per category
    seen = set()
    for r in records:
        if r["terrain_label"] not in seen:
            print(f"\n--- Example ({r['terrain_label']}) ---")
            print(json.dumps(r, indent=2))
            seen.add(r["terrain_label"])

Total pairs converted: 16000
✅ Saved to Outputs/crossmodal_fusion.jsonl

--- Example (agri) ---
{
  "id": "crossmodal_000000",
  "task": "fusion",
  "image_1": "/Users/farhanwajid/Web Development/SIH/v_2/agri/s2/ROIs1868_summer_s2_59_p10.png",
  "image_2": "/Users/farhanwajid/Web Development/SIH/v_2/agri/s1/ROIs1868_summer_s1_59_p10.png",
  "instruction": "Use the optical and SAR images together to identify the land cover type.",
  "response": "This region shows agricultural land, characterized by organized crop patterns visible in the optical image and moderate radar backscatter in the SAR image.",
  "bbox": null,
  "source_dataset": "Kaggle-Sentinel12",
  "terrain_label": "agri"
}

--- Example (barrenland) ---
{
  "id": "crossmodal_004000",
  "task": "fusion",
  "image_1": "/Users/farhanwajid/Web Development/SIH/v_2/barrenland/s2/ROIs1970_fall_s2_114_p1.png",
  "image_2": "/Users/farhanwajid/Web Development/SIH/v_2/barrenland/s1/ROIs1970_fall_s1_114_p1.png",
  "instruction": "What ty